# **Fase 5: MLOps y Produccion**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 12 febrero, 2026

# Notebook 11: Model Registry

**Conceptos clave:**
- **Model Registry**: Catálogo centralizado de modelos
- **Versioning**: Cada modelo puede tener múltiples versiones (v1, v2, etc.)
- **Stages**: Ciclo de vida: None -> Staging -> Production -> Archived
- **MlflowClient**: API programática para gestionar el registry

**Actividades:**

1. Registrar modelo en MLflow Model Registry
2. Crear versiones (v1, v2, etc.)
3. Transicionar entre stages: None -> Staging -> Production
4. Cargar modelo desde Registry

## 1. Configuración de SparkSession

In [8]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient

spark = SparkSession.builder \
    .appName("SECOP_ModelRegistry") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setLogLevel("WARN")

import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"


## 2. Reto 1: Configurar MLflow y MlflowClient

**Objetivo**: Conectar al tracking server y preparar el Model Registry.

**Instrucciones**:
1. Configura la URI del tracking server
2. Crea un `MlflowClient` para interactuar con el registry
3. Define un nombre descriptivo para tu modelo

**Pregunta**: ¿Qué diferencia hay entre el Tracking Server y el Model Registry?

El tracking server se enfoca en registrar ejecuciones individuales, incluyendo métricas, parámetros y artefactos de cada experimento. En cambio, el model registry gestiona modelos como entidades versionadas, permitiendo controlar su ciclo de vida, documentarlos y gobernar qué versión está activa en producción. Ambos se complementan: el tracking sirve para experimentar, el registry para operar modelos en entornos reales.


In [9]:
mlflow.set_tracking_uri("http://mlflow:5000")
client = MlflowClient()

model_name = "secop_prediccion_contratos"

print(f"MLflow URI: {mlflow.get_tracking_uri()}")
print(f"Modelo registrado: {model_name}")


MLflow URI: http://mlflow:5000
Modelo registrado: secop_prediccion_contratos


### 2.1 Cargar datos

In [10]:
df = spark.read.parquet("/opt/spark-data/processed/secop_ml_ready.parquet")
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_pca", "features") \
       .filter(col("label").isNotNull())

train, test = df.randomSplit([0.8, 0.2], seed=42)

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")

## 3. Reto 2: 	Entrenar y registrar modelo v1 (baseline)

**Objetivo**: Entrenar un modelo baseline y registrarlo como versión 1.

**Instrucciones**:
1. Configura el experimento
2. Entrena un modelo SIN regularización
3. Evalúa y registra métricas
4. Registra el modelo en el registry con `registered_model_name`

**Concepto clave**: Al usar `registered_model_name` en `log_model()`, MLflow automáticamente crea el modelo en el registry si no existe, o agrega una nueva versión si ya existe.

In [11]:
mlflow.set_experiment("/SECOP_Model_Registry")

with mlflow.start_run(run_name="model_v1_baseline") as run:
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        regParam=0.0,
        maxIter=100
    )
    
    model_v1 = lr.fit(train)
    predictions = model_v1.transform(test)
    rmse_v1 = evaluator.evaluate(predictions)

    mlflow.log_param("version", "1.0")
    mlflow.log_param("model_type", "baseline_no_regularization")
    mlflow.log_metric("rmse", rmse_v1)

    mlflow.spark.log_model(
        spark_model=model_v1,
        artifact_path="model",
        registered_model_name=model_name
    )

    run_id_v1 = run.info.run_id
    print(f"Modelo v1 registrado | Run ID: {run_id_v1}")
    print(f"RMSE v1: ${rmse_v1:,.2f}")


26/02/14 15:17:29 WARN Instrumentation: [a1441cfb] regParam is zero, which might cause numerical instability and overfitting.
2026/02/14 15:20:23 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpxo9420eo/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.
Registered model 'secop_prediccion_contratos' already exists. Creating a new version of this model...
2026/02/14 15:20:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: secop_prediccion_contratos, version 2
Created version '2' of model 'secop_prediccion_contratos'.


Modelo v1 registrado | Run ID: de55e7ad2a54413480846b450fe1fdb2
RMSE v1: $5,200,424,547.75


## 4. Reto 3: 	Entrenar y registrar modelo v2 (mejorado)

**Objetivo**: Entrenar un modelo mejorado y registrarlo como versión 2.

**Instrucciones**:
1. Entrena un modelo CON regularización (usa los mejores hiperparámetros del notebook 09)
2. Evalúa y compara con v1
3. Registra como nueva versión del mismo modelo

**Pregunta**: ¿Por qué versionar modelos en lugar de sobrescribir?

In [12]:
with mlflow.start_run(run_name="model_v2_regularized") as run:
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        regParam=0.1,
        elasticNetParam=0.5,
        maxIter=100
    )
    
    model_v2 = lr.fit(train)
    rmse_v2 = evaluator.evaluate(model_v2.transform(test))

    mlflow.log_param("version", "2.0")
    mlflow.log_param("model_type", "elasticnet_regularized")
    mlflow.log_metric("rmse", rmse_v2)

    mlflow.spark.log_model(
        spark_model=model_v2,
        artifact_path="model",
        registered_model_name=model_name
    )

    print(f"Modelo v2 registrado")
    print(f"RMSE v2: ${rmse_v2:,.2f}")


2026/02/14 15:25:51 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpkwvx6xnc/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.
26/02/14 15:25:51 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from jupyter:33021 in 10000 milliseconds
Registered model 'secop_prediccion_contratos' already exists. Creating a new version of this model...
2026/02/14 15:25:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: secop_prediccion_contratos, version 3


Modelo v2 registrado
RMSE v2: $5,200,424,547.67


Created version '3' of model 'secop_prediccion_contratos'.


### 3.1 Comparación

In [13]:
print("\nComparación de modelos:")
print(f"RMSE v1: ${rmse_v1:,.2f}")
print(f"RMSE v2: ${rmse_v2:,.2f}")
print(f"Mejor modelo: {'v2' if rmse_v2 < rmse_v1 else 'v1'}")



Comparación de modelos:
RMSE v1: $5,200,424,547.75
RMSE v2: $5,200,424,547.67
Mejor modelo: v2


**¿Por qué versionar modelos en lugar de sobrescribir?**

Porque permite trazabilidad y reproducibilidad. Sobrescribir un modelo elimina el historial y dificulta auditar decisiones o revertir cambios si una nueva versión falla en producción.

## 5. Reto 4: 	Gestionar stages: None → Staging → Production → Archived

**Objetivo**: Transicionar modelos entre stages del ciclo de vida.

**Ciclo de vida**:
 ```
 None -> Staging -> Production -> Archived
```
**Instrucciones**:
1. Lista las versiones registradas del modelo
2. Promueve la mejor versión a "Staging"
3. Si pasa la validación, promuévela a "Production"
4. Archiva la versión anterior

**¿Por qué pasar por Staging antes de Production?**

Staging permite validar el modelo en un entorno controlado, simulando condiciones reales sin impactar usuarios finales. Reduce riesgos, permite pruebas adicionales y asegura que solo modelos confiables lleguen a producción.

### 5.1 Listar versiones

In [15]:
model_versions = client.search_model_versions(f"name='{model_name}'")

print(f"Versiones del modelo '{model_name}':")
for mv in model_versions:
    print(f"- Versión {mv.version} | Stage={mv.current_stage} | Run={mv.run_id[:8]}")


Versiones del modelo 'secop_prediccion_contratos':
- Versión 3 | Stage=None | Run=8330f09e
- Versión 2 | Stage=None | Run=de55e7ad
- Versión 1 | Stage=None | Run=9240f8dc


### 5.2 Transiciones

In [16]:
client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Staging"
)
print("v1 → Staging")

if rmse_v2 < rmse_v1:
    client.transition_model_version_stage(
        name=model_name,
        version=2,
        stage="Production"
    )
    print("v2 → Production")

    client.transition_model_version_stage(
        name=model_name,
        version=1,
        stage="Archived"
    )
    print("v1 → Archived")


/tmp/ipykernel_20566/2352032923.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


v1 → Staging


/tmp/ipykernel_20566/2352032923.py:9: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


v2 → Production


/tmp/ipykernel_20566/2352032923.py:16: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


v1 → Archived


## 6. Reto 5: 	Agregar metadata y descripcion al modelo

**Objetivo**: Documentar el modelo con descripciones y etiquetas.

**Instrucciones**:
1. Agrega una descripción a la versión en producción
2. Incluye información útil: RMSE, fecha, autor, dataset usado

**Pregunta**: ¿Qué información mínima debería tener cada versión de modelo?

Métricas clave (RMSE, MAE, R²), tipo de modelo, dataset utilizado, hiperparámetros relevantes y propósito del modelo.


In [17]:
best_version = 2 if rmse_v2 < rmse_v1 else 1

client.update_model_version(
    name=model_name,
    version=best_version,
    description=(
        f"Modelo ElasticNet para predicción de contratos SECOP. "
        f"RMSE: ${min(rmse_v1, rmse_v2):,.2f}. "
        f"Dataset: secop_ml_ready.parquet. "
        f"Entrenado en Feb 2026."
    )
)

print(f"Metadata agregada a la versión {best_version}")


Metadata agregada a la versión 2


## 7. Reto 6: 	Cargar modelo desde Registry para prediccion

**Objetivo**: Cargar el modelo en producción para hacer predicciones.

**Concepto**: En producción, cargamos modelos por su nombre y stage, NO por ruta de archivo. Esto permite:
- Cambiar la versión en producción sin modificar código
- Rollback instantáneo si algo falla

**Instrucciones**:
1. Carga el modelo desde el Registry usando `models:/{name}/{stage}`
2. Verifica que funciona haciendo predicciones en test
3. Compara el RMSE con el esperado

In [18]:
model_uri = f"models:/{model_name}/Production"
loaded_model = mlflow.spark.load_model(model_uri)

print(f"Modelo cargado desde: {model_uri}")
print(f"Tipo de modelo: {type(loaded_model)}")

test_predictions = loaded_model.transform(test)
test_rmse = evaluator.evaluate(test_predictions)

print(f"RMSE verificación producción: ${test_rmse:,.2f}")


/usr/local/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.9.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])
2026/02/14 15:57:18 INFO mlflow.spark: 'models:/secop_prediccion_contratos/Production' resolved as 'file:///opt/mlflow/mlruns/324944293645910354/de55e7ad2a54413480846b450fe1fdb2/artifacts/model'
2026/02/14 15:57:21 INFO mlflow.spark: URI 'models:/secop_prediccion_contratos/Production/sparkml' does not point to the current DFS.
2026/02/14 15:57:21 INFO mlflow.spark: File 'models:/secop_prediccion_contratos/Production/sparkml' not found on DFS. Will attempt to upload the file.


Modelo cargado desde: models:/secop_prediccion_contratos/Production
Tipo de modelo: <class 'pyspark.ml.pipeline.PipelineModel'>


RMSE verificación producción: $5,200,424,547.75


## 8. Preguntas de reflexión

**¿Cómo harías rollback si el modelo en Production falla?**

*Respuesta:* Revirtiendo el stage de production a una versión anterior estable directamente desde el model registry, sin modificar código ni redeplegar servicios.

**¿Qué criterios usarías para promover un modelo de Staging a Production?**

*Respuesta:* Mejor rendimiento en métricas clave, estabilidad en staging y validación con datos recientes.

**¿Cómo implementarías A/B testing con el Model Registry?**

*Respuesta:* Asignando diferentes versiones del modelo a subconjuntos de tráfico y comparando métricas antes de promover una versión definitiva.

**¿Quién debería tener permisos para promover modelos a Production?**

*Respuesta:* Roles como ML Engineers o Tech Leads, siguiendo un proceso de revisión.

In [19]:

print("Resumen model")

print("Verifica que hayas completado:")
print("  [✓] Registrado modelo v1 (baseline)")
print("  [✓] Registrado modelo v2 (mejorado)")
print("  [✓] Transicionado versiones entre stages")
print("  [✓] Agregado metadata al modelo")
print("  [✓] Cargado modelo desde Registry")
print("  [✓] Accede a Model Registry: http://localhost:5000/#/models")


Resumen model
Verifica que hayas completado:
  [✓] Registrado modelo v1 (baseline)
  [✓] Registrado modelo v2 (mejorado)
  [✓] Transicionado versiones entre stages
  [✓] Agregado metadata al modelo
  [✓] Cargado modelo desde Registry
  [✓] Accede a Model Registry: http://localhost:5000/#/models


In [20]:
spark.stop()